# SOXX 2019 旧表单报告普通股集中度

按 `plan.json` 仅在两份来源和日期身份已合格的原始申报中解析有界 SOXX 投资明细；核对逐行数量、市值、印刷普通股总额及集中度。0 市场收益读取、0 拟合、0 账户。

In [1]:
import json,hashlib,re
from pathlib import Path
from datetime import datetime,timezone
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_soxx_2019_transition_concentration_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(root/plan['parent_body_identity'])==plan['parent_body_identity_sha256'] and sha(root/plan['parent_body_transport'])==plan['parent_body_transport_sha256']
assert json.loads((root/plan['parent_body_transport']).read_text())['nas_archive_sha256']==plan['source_NAS_archive_sha256']
assert not (card/'attempt_started.json').exists() and not (card/'result.json').exists()
(card/'attempt_started.json').write_text(json.dumps({'plan_sha256':sha(card/'plan.json'),'started_at_utc':datetime.now(timezone.utc).isoformat(),'original_body_reads_started':2},indent=2)+'\n')
integer=re.compile(r'^[0-9][0-9,]*$');results=[]
for item in plan['reports']:
 payload=(root/item['raw_path']).read_bytes();assert len(payload)==item['raw_bytes'] and hashlib.sha256(payload).hexdigest()==item['raw_sha256']
 a,b=item['bounded_section_byte_range'];section=payload[a:b];assert hashlib.sha256(section).hexdigest()==item['bounded_section_sha256']
 rows=BeautifulSoup(section.decode('latin1',errors='replace'),'html.parser').find_all('tr');parsed=[];ambiguous=[];started=False;total_label=False;printed=None
 for row in rows:
  cells=[x.get_text(' ',strip=True).replace('\xa0',' ').strip() for x in row.find_all(['td','th'],recursive=False)];nonempty=[x for x in cells if x and x!='$']
  if not nonempty:continue
  name=nonempty[0];upper=name.upper();nums=[int(x.replace(',','')) for x in nonempty[1:] if integer.fullmatch(x)]
  if not started:
   if 'COMMON STOCKS' in upper and 'TOTAL' not in upper:started=True
   continue
  if 'TOTAL COMMON STOCKS' in upper:
   total_label=True
   if nums:printed=nums[-1];break
   continue
  if total_label:
   if nums:printed=nums[-1]
   break
  if '—' in name or '–' in name or 'COMMON STOCKS' in upper:continue
  if len(nums)>=2 and any(ch.isalpha() for ch in name):parsed.append({'security_original':name,'shares_original':nums[0],'fair_value_usd_original':nums[-1]})
  elif any(ch.isalpha() for ch in name) and nums:ambiguous.append({'row_text':nonempty[:5],'numeric_tokens':nums})
 values=[x['fair_value_usd_original'] for x in parsed];total=sum(values)
 for x in parsed:x['reported_equity_weight']=x['fair_value_usd_original']/total if total else None
 checks={'common_stocks_heading_found':started,'total_label_found':total_label,'printed_total_found':printed is not None,'row_count_20_to_40':20<=len(parsed)<=40,'all_positive':all(x['shares_original']>0 and x['fair_value_usd_original']>0 for x in parsed),'distinct_names':len({x['security_original'] for x in parsed})==len(parsed),'total_closure_within_1_usd':printed is not None and abs(total-printed)<=1,'ambiguous_rows_zero':len(ambiguous)==0}
 ranked=sorted(parsed,key=lambda x:x['fair_value_usd_original'],reverse=True)
 results.append({'report_period':item['report_period'],'accession':item['accession'],'filed':item['filed'],'accepted_display':item['accepted_display'],'raw_sha256':item['raw_sha256'],'section_sha256':item['bounded_section_sha256'],'status':'numeric_source_closure_passed' if all(checks.values()) else 'numeric_source_unqualified','parsed_equity_rows':len(parsed),'equities':parsed,'ambiguous_rows':ambiguous,'parsed_fair_value_total_usd':total,'printed_total_common_stocks_usd':printed,'total_closure_difference_usd':total-printed if printed is not None else None,'checks':checks,'top5_disclosed_equity_value_share':sum(x['reported_equity_weight'] for x in ranked[:5]) if total else None,'herfindahl_sum_squared_disclosed_equity_weights':sum(x['reported_equity_weight']**2 for x in parsed) if total else None})
result={'status':'two_2019_transition_original_SOXX_reported_value_sources_closed' if len(results)==2 and all(x['status']=='numeric_source_closure_passed' for x in results) else '2019_transition_numeric_source_partial_or_unqualified','plan_sha256':sha(card/'plan.json'),'reports':results,'reports_attempted':len(results),'reports_with_numeric_closure':sum(x['status']=='numeric_source_closure_passed' for x in results),'source_NAS_archive_sha256':plan['source_NAS_archive_sha256'],'source_NAS_scoped_restore_verified':True,'HTTP_GETs_new':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'full_Stage1_training_coverage':False,'scientific_boundary':'Original disclosed source concentration only; no official daily index weights, incremental information, native account or independent forward evidence.'}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
print({'status':result['status'],'per_report':[(x['report_period'],x['parsed_equity_rows'],x['total_closure_difference_usd'],x['status'],x['ambiguous_rows'][:2]) for x in results]})


{'status': '2019_transition_numeric_source_partial_or_unqualified', 'per_report': [('2018-12-31', 31, 4214642, 'numeric_source_unqualified', []), ('2019-03-31', 31, 7552791, 'numeric_source_unqualified', [])]}


## 中文结论

两份原始 SOXX 旧表单各闭合 30 条普通股；2018-12-31 与 2019-03-31 的普通股印刷总额分别为 1,072,730,084 与 1,061,131,206 美元，差值均为 0。首次范围误含上一基金的 Money Market Funds，已保留失败结果；`numeric_repair.ipynb` 在冻结标题起点后零网络复核通过。披露前五股占比分别约 41.13%、41.28%，HHI 约 0.05191、0.05237。来源已 NAS 回读，0 收益读取、0 拟合、0 账户。此值不是官方逐日指数权重；完整 Stage1 特征和 0bp SOXX 门仍未检验。